# Tutorial: Reproducing the Emails CBM Experiment

This notebook reproduces the full pipeline for the phishing-email Concept
Bottleneck Model (CBM), from raw data to a trained, evaluated model:

1. Load the raw email corpus
2. Filter to the "usable" subset (language, length, authorship)
3. Encode email bodies with a sentence-transformer
4. Build concept ground truth and the task label
5. Split into train / validation / test (matching the real study)
6. Train the concept extractor (embedding -> concepts)
7. Train the label predictor (concepts -> task label)
8. Evaluate end-to-end, with classification reports at every stage
9. **Sanity check**: verify the reproduced data matches the originally
   published dataset (`NWeak/emails-mirror` on HuggingFace)

**Out of scope**: this notebook does *not* touch the actual human user
study (participant responses, trust/confidence analysis, etc.) -- see
`emails/PARTICIPANT_DATA_REPORT.md` and `SUPPLEMENTARY_ANALYSIS.md` for
that. For the full written explanation of every step below, see
`emails/emails_preprocessing.md` and `supplementary_materials.tex`
(repo root).

Run this notebook from `emails/scripts/` (open it directly in Jupyter, or
execute headlessly via nbclient).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from langdetect import DetectorFactory, detect
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

DetectorFactory.seed = 0  # deterministic langdetect results

/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the raw corpus

The raw corpus is a JSONL file, one email per line, with fields including
`Body`, `Subject`, `Type` (Valid / Phishing / Phishing Simulation / Spam),
`Created by` (Human / other), `LLM detected emotion`, `LLM detected motivation`.

In [2]:
raw_path = Path('../data/merged_emails_with_categories.jsonl')
with open(raw_path) as f:
    original = pd.read_json(f, lines=True)

print(f"Loaded {len(original)} raw emails")
original[['Subject', 'Type', 'Created by']].head()

Loaded 12300 raw emails


,Subject,Type,Created by
0,New login on your pCloud account,Phishing,Human
1,INVITATION TO THE 79TH IFALPA ANNUAL CONFERENCE,Phishing,Human
2,bitcoin payment,Phishing,Human
3,DHL Megjegyzés: A csomag kiadásához vámkezelé...,Phishing,Human
4,Quote Request - Mol Group,Phishing,Human


## 2. Annotate language and word count

Not a filter yet -- just computes two new columns (`lang`, `words`) that
the next step filters on. Word count is a plain whitespace split; language
is `langdetect`'s own guess, seeded for determinism.

In [3]:
def analyze_email_body(text):
    if not text or text.strip() == "":
        return {"word_count": 0, "language": "unknown"}
    word_count = len(text.split())
    try:
        language = detect(text)
    except Exception:
        language = "could not determine"
    return {"word_count": word_count, "language": language}

languages, words = [], []
for body in original['Body']:
    result = analyze_email_body(body)
    languages.append(result['language'])
    words.append(int(result['word_count']))

original['lang'] = languages
original['words'] = words
print(original['lang'].value_counts().head())

lang
en    11890
no       97
ja       65
pt       47
fr       42
Name: count, dtype: int64


## 3. Filter to the "usable" set

Three conditions applied jointly: English language, under 400 words, and
human-authored (excludes LLM-generated/synthetic rows). This is the only
row-level filtering in the emails pipeline -- unlike CUB, which uses every
image unmodified (see `cub/cub_preprocessing.md`).

In [4]:
usable = original[
    (original['lang'] == 'en')
    & (original['words'] < 400)
    & (original['Created by'] == 'Human')
].reset_index(drop=True)

print(f"{len(original)} raw emails -> {len(usable)} usable emails")
assert len(usable) == 2330, f"expected 2330 usable emails, got {len(usable)}"

12300 raw emails -> 2330 usable emails


## 4. Encode with sentence-transformers

Only `Body` is encoded (never `Subject`), as-is -- no HTML stripping,
lowercasing, or extra truncation beyond the word-count filter above. This
runs once, over the entire usable set, before any train/val/test split
exists.

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(usable['Body'].tolist(), show_progress_bar=True)
usable['embedding'] = list(embeddings)
print(f"Encoded {len(usable)} emails to {embeddings.shape[1]}-dim vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12027.10it/s]

Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Batches:   1%|▏         | 1/73 [00:01<01:12,  1.00s/it]

Batches:   5%|▌         | 4/73 [00:01<00:14,  4.61it/s]

...

Batches:  81%|████████  | 59/73 [00:02<00:00, 35.20it/s]

Batches:  88%|████████▊ | 64/73 [00:02<00:00, 38.84it/s]

Batches: 100%|██████████| 73/73 [00:03<00:00, 52.51it/s]

Batches: 100%|██████████| 73/73 [00:03<00:00, 24.17it/s]

Encoded 2330 emails to 384-dim vectors


## 5. Build concept ground truth and task label

The 19-dim raw concept vector (`concept_gts`) comes from LLM-labeled
emotion/motivation fields. It's reduced to a 6-dim filtered vector
(`concept_gts_f`) matching the six concepts actually used by the model and
shown to human participants: **Fear+Authority** (a merge of two raw
concepts), **Urgency**, **Curiosity**, **Neutral**, **Reply**,
**Open attachment**. The task label comes from `Type`: `Valid` -> 1,
`Phishing`/`Phishing Simulation` -> 0, `Spam` -> 2 (label 2 is excluded
from CBM training/evaluation below -- the task is binary valid-vs-phishing).

`LLM detected emotion`/`LLM detected motivation` are already Python lists
(this dataframe came straight from `pd.read_json`, not a CSV round-trip --
if reading from a CSV instead, these columns would need `ast.literal_eval`
first to parse the string representation back into a list).

In [6]:
all_emotions, all_motivations = [], []
for i in range(len(usable)):
    for e in usable['LLM detected emotion'].iloc[i]:
        if e not in all_emotions:
            all_emotions.append(e)
    for m in usable['LLM detected motivation'].iloc[i]:
        if m not in all_motivations:
            all_motivations.append(m)

concept_gts = []
for i in range(len(usable)):
    emotions = usable['LLM detected emotion'].iloc[i]
    motivations = usable['LLM detected motivation'].iloc[i]
    row = [1 if e in emotions else 0 for e in all_emotions]
    row += [1 if m in motivations else 0 for m in all_motivations]
    concept_gts.append(row)

usable['concept_gts'] = concept_gts
print(f"{len(all_emotions) + len(all_motivations)}-dim raw concept vector "
      f"({len(all_emotions)} emotions + {len(all_motivations)} motivations)")

19-dim raw concept vector (12 emotions + 7 motivations)


In [7]:
FEAR_IDX = all_emotions.index('Fear')
AUTHORITY_IDX = all_emotions.index('Authority')
filtered_idx = [
    all_emotions.index('Urgency'), all_emotions.index('Curiosity'), all_emotions.index('Neutral'),
    len(all_emotions) + all_motivations.index('Reply'),
    len(all_emotions) + all_motivations.index('Open attachment'),
]

labels, concept_gts_f = [], []
for i in range(len(usable)):
    email_type = usable['Type'].iloc[i]
    if email_type == 'Valid':
        labels.append(1)
    elif email_type in ('Phishing', 'Phishing Simulation'):
        labels.append(0)
    elif email_type == 'Spam':
        labels.append(2)
    else:
        raise NotImplementedError(email_type)

    concepts = usable['concept_gts'].iloc[i]
    fear_or_authority = 1 if (concepts[FEAR_IDX] == 1 or concepts[AUTHORITY_IDX] == 1) else 0
    concept_gts_f.append([fear_or_authority] + [concepts[j] for j in filtered_idx])

usable['label'] = labels
usable['concept_gts_f'] = concept_gts_f

with open('../metadata/emails/concept_names_filtered.txt') as f:
    CONCEPT_NAMES = f.read().splitlines()
print("Filtered concepts:", CONCEPT_NAMES)

Filtered concepts: ['Fear+Authority', 'Urgency', 'Curiosity', 'Neutral', 'Reply', 'Open attachment']


## 6. Train / validation / test split

A naive stochastic split (`train_test_split(random_state=42)`) turns out
**not to be reproducible** across pandas/numpy versions -- two different
environments with the same seed produced different splits. The
authoritative split instead selects rows *by fixed email ID* from a
pre-recorded list matching the real study (`archival_split_ids.json`). The
embeddings/concepts above were computed once, before this split -- the
split only selects which already-processed rows go into each file.

In [8]:
with open('../metadata/emails/archival_split_ids.json') as f:
    archival_ids = json.load(f)

final = usable
train = final[final['Original email No.'].isin(archival_ids['train'])].reset_index(drop=True)
val = final[final['Original email No.'].isin(archival_ids['val'])].reset_index(drop=True)
test = final[final['Original email No.'].isin(archival_ids['test'])].reset_index(drop=True)

print(f"train={len(train)}, val={len(val)}, test={len(test)}")
train_class = train[train['label'].isin([0, 1])]
test_class = test[test['label'].isin([0, 1])]
print(f"train_class={len(train_class)} (rows with a binary task label), test_class={len(test_class)}")

train=1064, val=266, test=1000
train_class=551 (rows with a binary task label), test_class=1000


## 7. Train the concept extractor

One binary linear SVM per concept (six independent classifiers via
scikit-learn's `MultiOutputClassifier`), trained on the *full* training
split (concept labels are defined for every email, including `Spam`).

In [9]:
X_train = np.stack(train['embedding'].values)
y_train_concepts = np.stack(train['concept_gts_f'].values)

concept_model = MultiOutputClassifier(SVC(kernel='linear', C=1.0))
concept_model.fit(X_train, y_train_concepts)
print("Concept extractor trained.")

Concept extractor trained.


### Concept extractor: classification report on the test set

In [10]:
X_test = np.stack(test_class['embedding'].values)
concept_preds = concept_model.predict(X_test)
y_test_concepts = np.stack(test_class['concept_gts_f'].values)
print(classification_report(y_test_concepts, concept_preds, target_names=CONCEPT_NAMES))

                 precision    recall  f1-score   support

 Fear+Authority       0.89      0.91      0.90       501
        Urgency       0.88      0.94      0.91       553
      Curiosity       0.62      0.50      0.55       262
        Neutral       0.94      0.82      0.88       416
          Reply       0.89      0.24      0.38        33
Open attachment       0.71      0.07      0.14        67

      micro avg       0.87      0.80      0.83      1832
      macro avg       0.82      0.58      0.63      1832
   weighted avg       0.86      0.80      0.81      1832
    samples avg       0.86      0.82      0.82      1832



/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 8. Train the label predictor (independently, on ground-truth concepts)

A single logistic regression, trained *only* on training rows with a
binary task label, using ground-truth concepts rescaled from {0,1} to
{-1,+1}. `fit_intercept=False` makes the fitted weight vector directly
interpretable as each concept's signed contribution; `penalty=None`
(unregularized) is viable given the tiny 6-dim input; `class_weight='balanced'`
compensates for the training split's class imbalance.

In [11]:
train_concepts_pm1 = 2 * np.stack(train_class['concept_gts_f'].values) - 1
train_labels = np.stack(train_class['label'].values)

label_model = LogisticRegression(max_iter=1000, C=1, solver='lbfgs',
                                  fit_intercept=False, penalty=None, class_weight='balanced')
label_model.fit(train_concepts_pm1, train_labels)
print("Label predictor trained.")

Label predictor trained.


/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


### Label predictor: classification report, evaluated directly on ground-truth test concepts

This is the accuracy **upper bound** -- bypassing the concept extractor entirely.

In [12]:
test_concepts_pm1 = 2 * np.stack(test_class['concept_gts_f'].values) - 1
test_labels = np.stack(test_class['label'].values)

y_pred_gt = label_model.predict(test_concepts_pm1)
print(classification_report(test_labels, y_pred_gt))

              precision    recall  f1-score   support

           0       0.93      0.92      0.93       500
           1       0.92      0.94      0.93       500

    accuracy                           0.93      1000
   macro avg       0.93      0.93      0.93      1000
weighted avg       0.93      0.93      0.93      1000



## 9. End-to-end evaluation

At inference time the two stages are chained: an email's embedding goes
through each concept SVM's *decision function* (continuous distance to the
hyperplane, not the binarized prediction), through `tanh` to bound it to
(-1, 1), and the resulting 6-dim vector is fed to the label predictor.

In [13]:
logits = np.column_stack([est.decision_function(X_test) for est in concept_model.estimators_])
concept_activations = np.tanh(logits)
y_pred_e2e = label_model.predict(concept_activations)

print(classification_report(test_labels, y_pred_e2e))
acc = accuracy_score(test_labels, y_pred_e2e)
print(f"End-to-end accuracy = {acc:.4f}")

              precision    recall  f1-score   support

           0       0.91      0.94      0.92       500
           1       0.93      0.91      0.92       500

    accuracy                           0.92      1000
   macro avg       0.92      0.92      0.92      1000
weighted avg       0.92      0.92      0.92      1000

End-to-end accuracy = 0.9230


## 10. Sanity check: does the reproduced data match the original?

Everything above was recomputed from scratch (raw JSONL -> filtered ->
encoded -> split). This compares the reproduced train/val/test data
against the originally published `NWeak/emails-mirror` dataset on
HuggingFace, to confirm this notebook is a faithful reproduction, not just
"a" pipeline that happens to run.

In [14]:
from datasets import load_dataset

hf = load_dataset("NWeak/emails-mirror")

mismatches = []
for split_name, local_df in [("train", train), ("val", val), ("test", test)]:
    hf_df = hf[split_name].to_pandas().set_index("Original email No.")
    local_indexed = local_df.set_index("Original email No.")

    if set(local_indexed.index) != set(hf_df.index):
        mismatches.append(f"{split_name}: ID sets differ "
                           f"(local-only={len(set(local_indexed.index) - set(hf_df.index))}, "
                           f"hf-only={len(set(hf_df.index) - set(local_indexed.index))})")
        continue

    hf_df = hf_df.loc[local_indexed.index]
    local_emb = np.stack(local_indexed['embedding'].values)
    hf_emb = np.stack(hf_df['embedding'].values)
    if not np.allclose(local_emb, hf_emb, atol=1e-4):
        mismatches.append(f"{split_name}: embeddings differ beyond tolerance "
                           f"(max abs diff {np.abs(local_emb - hf_emb).max():.2e})")

    local_concepts = np.stack(local_indexed['concept_gts_f'].values)
    hf_concepts = np.stack(hf_df['concept_gts_f'].values)
    if not np.array_equal(local_concepts, hf_concepts):
        mismatches.append(f"{split_name}: concept_gts_f differ ({(local_concepts != hf_concepts).sum()} cells)")

    local_labels = local_indexed['label'].values
    hf_labels = hf_df['label'].values
    if not np.array_equal(local_labels, hf_labels):
        mismatches.append(f"{split_name}: label differs ({(local_labels != hf_labels).sum()} rows)")

    print(f"[{split_name}] {len(local_indexed)} rows checked against NWeak/emails-mirror: OK")

assert not mismatches, "Reproduced data does not match NWeak/emails-mirror:\n" + "\n".join(mismatches)
print("\nPASSED: reproduced train/val/test data (IDs, embeddings, concepts, labels) "
      "matches the originally published NWeak/emails-mirror dataset.")

[train] 1064 rows checked against NWeak/emails-mirror: OK
[val] 266 rows checked against NWeak/emails-mirror: OK
[test] 1000 rows checked against NWeak/emails-mirror: OK

PASSED: reproduced train/val/test data (IDs, embeddings, concepts, labels) matches the originally published NWeak/emails-mirror dataset.
